# 03 — Data Cleaning & Leakage Hunt

Two jobs in this notebook:
1. **Fix the data** — correct types, handle missing values.
2. **Prevent leakage** — drop columns that secretly encode the answer, so the
   model learns from real physical signal, not from NASA's own verdict.

Loads the raw CSV fresh (not the SQLite DB) so pandas re-infers proper numeric
types, fixing the everything-as-text issue from notebook 02.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Load raw data straight from the CSV so pandas infers numeric types correctly.
df = pd.read_csv("../data/raw/koi_cumulative.csv")
print("Shape:", df.shape)
df.head()

Shape: (9564, 153)


,kepid,kepoi_name,kepler_name,ra,ra_err,ra_str,dec,dec_err,dec_str,koi_gmag,...,koi_fpflag_co,koi_fpflag_ec,koi_insol,koi_insol_err1,koi_insol_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype,koi_score
0,10797460,K00752.01,Kepler-227 b,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,93.59,29.45,-16.65,3.20796,0.33173,-1.09986,LS+MCMC,1.000
1,10797460,K00752.02,Kepler-227 c,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,9.11,2.87,-1.62,3.02368,2.20489,-2.49638,LS+MCMC,0.969
2,10811496,K00753.01,NaN,297.00482,0.0,19h48m01.16s,48.134129,0.0,+48d08m02.9s,15.943,...,0,0,39.30,31.04,-10.49,7.29555,35.03293,-2.75453,LS+MCMC,0.000
3,10848459,K00754.01,NaN,285.53461,0.0,19h02m08.31s,48.285210,0.0,+48d17m06.8s,16.100,...,0,0,891.96,668.95,-230.35,0.22080,0.00917,-0.01837,LS+MCMC,0.000
4,10854555,K00755.01,Kepler-664 b,288.75488,0.0,19h15m01.17s,48.226200,0.0,+48d13m34.3s,16.015,...,0,0,926.16,874.33,-314.24,1.98635,2.71141,-1.74541,LS+MCMC,1.000


In [3]:
# Count missing values per column, show the 20 emptiest.
# Now that types are numeric again, real NaNs are detected properly.
df.isnull().sum().sort_values(ascending=False).head(20)

koi_sma_err2        9564
koi_incl_err2       9564
koi_kepmag_err      9564
koi_incl_err1       9564
koi_model_dof       9564
koi_model_chisq     9564
koi_ingress_err2    9564
koi_sage_err2       9564
koi_eccen_err1      9564
koi_eccen_err2      9564
koi_zmag_err        9564
koi_longp           9564
koi_imag_err        9564
koi_longp_err1      9564
koi_rmag_err        9564
koi_longp_err2      9564
koi_gmag_err        9564
koi_sage_err1       9564
koi_sage            9564
koi_ingress_err1    9564
dtype: int64

In [4]:
# Missing percentage per column, highest first
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(f"{len(missing_pct)} columns have missing data\n")
missing_pct

127 columns have missing data



koi_incl_err1       100.0
koi_ingress_err2    100.0
koi_eccen_err1      100.0
koi_eccen_err2      100.0
koi_longp           100.0
                    ...  
koi_gmag              0.4
koi_hmag              0.3
koi_jmag              0.3
koi_kmag              0.3
koi_rmag              0.1
Length: 127, dtype: float64

In [5]:
# Rule: drop columns missing more than 50% of values — too empty to impute
# reliably. The data splits cleanly (either ~100% missing or <1%), so 50%
# is a safe dividing line. Mostly the _err1/_err2 uncertainty columns.
threshold = 50   # the percentage cutoff
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns over {threshold}% missing")
df = df.drop(columns=cols_to_drop)
print("Shape after dropping empty columns:", df.shape)

Dropping 25 columns over 50% missing
Shape after dropping empty columns: (9564, 128)


## Leakage Hunt

The columns above were dropped for being *empty*. These next columns are
dropped for a different reason: they leak the answer. Each one either IS the
target, encodes NASA's own verdict, or is an identifier with no physical
meaning. Training on them would inflate accuracy while teaching the model
nothing real — it'd be reading the answer key instead of the physics.

In [6]:
# Each column dropped for a specific leakage/uselessness reason:
leakage_cols = {
    "koi_pdisposition": "Preliminary disposition from the Kepler pipeline — an earlier version of the target label, so it directly leaks the answer.",
    "koi_score":        "NASA's own confidence (0-1) that the object is a planet; tracks the disposition almost perfectly (seen in nb 01). Pure leakage.",
    "koi_fpflag_nt":    "False-positive flag (not transit-like) — part of NASA's vetting verdict, derived from the label itself.",
    "koi_fpflag_ss":    "False-positive flag (stellar eclipse) — same: a vetting conclusion, not a raw measurement.",
    "koi_fpflag_co":    "False-positive flag (centroid offset) — vetting verdict, leaks the answer.",
    "koi_fpflag_ec":    "False-positive flag (ephemeris match/contamination) — vetting verdict, leaks the answer.",
}

identifier_cols = {
    "kepid":      "Catalog ID number — a label with no physical meaning; the value itself tells you nothing about whether the object is a planet.",
    "kepoi_name": "Catalog name/identifier, same as kepid — an arbitrary label, not a measurement.",
    "ra":  "Right ascension (sky position). Whether a dip is a planet vs eclipsing binary is a property of the star system, not its location on the sky — so position shouldn't carry predictive signal. Dropping, though open to revisiting.",
    "dec": "Declination (sky position), same reasoning as ra — location shouldn't determine planet-hood.",
}

cols_to_drop = list(leakage_cols) + list(identifier_cols)
# only drop ones still present (some may already be gone, e.g. kepler_name)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} leakage/identifier columns")
print("Shape:", df.shape)

Dropped 10 leakage/identifier columns
Shape: (9564, 118)


In [7]:
# ra_str / dec_str are string-formatted sky coordinates — same useless
# position info as ra/dec (already dropped), just as text. Remove them too.
coord_strings = [c for c in ["ra_str", "dec_str", "ra_err", "dec_err"] if c in df.columns]
df = df.drop(columns=coord_strings)
print(f"Dropped {len(coord_strings)} leftover coordinate columns")
print("Shape:", df.shape)

Dropped 4 leftover coordinate columns
Shape: (9564, 114)


In [8]:
# Remaining missing data, now that empty/leakage columns are gone
remaining_missing = (df.isnull().sum() / len(df) * 100).round(2)
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"{len(remaining_missing)} columns still have some missing data")
remaining_missing

102 columns still have some missing data


koi_bin_oedp_sig    15.79
koi_comment         12.64
koi_max_sngle_ev    11.94
koi_quarters        11.94
koi_max_mult_ev     11.94
                    ...  
koi_kmag             0.26
koi_hmag             0.26
koi_jmag             0.26
koi_rmag             0.09
koi_kepmag           0.01
Length: 102, dtype: float64

In [9]:
df["koi_comment"].dtype, df["koi_comment"].head()

(<StringDtype(storage='python', na_value=nan)>,
 0                                          NO_COMMENT
 1                                          NO_COMMENT
 2                                       DEEP_V_SHAPED
 3    MOD_ODDEVEN_DV---MOD_ODDEVEN_ALT---DEEP_V_SHAPED
 4                                          NO_COMMENT
 Name: koi_comment, dtype: str)

In [10]:
# koi_comment is free-text vetting notes — can't median-impute text, out of
# scope for a numeric classifier, and possibly leaky (notes can hint the verdict).
if "koi_comment" in df.columns:
    df = df.drop(columns=["koi_comment"])

# Median-impute remaining numeric gaps: fill rather than drop rows so we keep
# otherwise-complete observations. Median (not mean) is robust to the outliers
# seen in nb 02 (e.g. inflated false-positive radii won't skew the fill).
numeric_cols = df.select_dtypes(include="number").columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Total missing values remaining:", df.isnull().sum().sum())
print("Shape:", df.shape)

Total missing values remaining: 3269
Shape: (9564, 113)


In [11]:
df.isnull().sum()[df.isnull().sum() > 0]

koi_quarters         1142
koi_limbdark_mod      363
koi_trans_mod         363
koi_sparprov          363
koi_tce_delivname     346
koi_datalink_dvs      346
koi_datalink_dvr      346
dtype: int64

In [12]:
# Remaining missing columns are all text metadata/identifiers — observing
# quarters, model names, parameter provenance, pipeline delivery labels.
# None describe the physical object, so none carry predictive signal. Drop them.
metadata_cols = [
    "koi_quarters", "koi_limbdark_mod", "koi_trans_mod", "koi_sparprov",
    "koi_tce_delivname", "koi_datalink_dvs", "koi_datalink_dvr",
]
metadata_cols = [c for c in metadata_cols if c in df.columns]
df = df.drop(columns=metadata_cols)

print(f"Dropped {len(metadata_cols)} text metadata columns")
print("Total missing values remaining:", df.isnull().sum().sum())
print("Shape:", df.shape)

Dropped 7 text metadata columns
Total missing values remaining: 0
Shape: (9564, 106)


In [13]:
# Save the cleaned, model-ready dataset for the modeling notebook.
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "koi_cleaned.csv", index=False)
print(f"Saved cleaned data: {df.shape} -> {out_dir / 'koi_cleaned.csv'}")

Saved cleaned data: (9564, 106) -> ../data/processed/koi_cleaned.csv


## Summary

Started from the raw KOI table (9,564 × 153) and produced a clean,
model-ready dataset through three kinds of decisions:

**1. Dropped empty columns (>50% missing).** 25 columns — almost all the
`_err1`/`_err2` measurement-uncertainty fields — were over half empty, too
sparse to impute reliably. Used a 50% threshold, which sat cleanly in the gap
between ~100%-missing and <1%-missing columns.

**2. Dropped leakage and identifier columns.** Removed columns that would let
the model cheat or that carry no physical signal:
- *Leakage:* `koi_score`, `koi_pdisposition`, and the four `koi_fpflag_*`
  flags — all derived from NASA's own vetting verdict, i.e. the answer.
- *Identifiers / location:* `kepid`, `kepoi_name`, `ra`, `dec` (and their
  `_str`/`_err` variants) — labels and sky positions with no bearing on
  whether a signal is a planet.
- *Text metadata:* observing quarters, model names, parameter provenance, and
  pipeline delivery labels — bookkeeping about the data, not the object.

**3. Imputed the rest.** Remaining numeric gaps (all <16% missing, including
the 363 missing radii found in notebook 02) were filled with the column
**median** — robust to the outliers seen earlier — rather than dropping rows
and losing otherwise-complete observations.

**Result:** a clean dataset with 0 missing values, all numeric features plus
the `koi_disposition` target, saved to `data/processed/koi_cleaned.csv`.

**Guiding principle:** every column was judged by one question — *is this a
property of the object (keep) or information about how the data was produced
(drop)?* — and every drop and fill is documented above so the choices can be
defended, not just reproduced.

**Next:** feature engineering and modeling (notebook 04), where the kept
physical features (period, radius, stellar temperature, etc.) are used to
classify each KOI.